# polls_question 기초 전처리

결측·이상치·중복을 확인합니다. 질문 내용 중복은 확인 결과로 남기며, 행 삭제와 CSV 저장은 수행하지 않습니다.


## 1. 공통 준비

현재 실행 폴더부터 상위 폴더를 확인해 프로젝트 루트를 찾고 팀의 DB 연결 함수를 불러옵니다.


In [248]:
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display
from sqlalchemy import text
from dotenv import load_dotenv
def convert_column_types(
    df,
    object_columns,
    datetime_columns,
    table_name
):
    """
    지정한 컬럼을 문자형(object)과 datetime으로 변환합니다.
    datetime 변환에 실패한 값은 NaT로 처리하고 개수를 출력합니다.
    """

    required_columns = object_columns + datetime_columns
    missing_columns = [
        column for column in required_columns
        if column not in df.columns
    ]

    if missing_columns:
        raise KeyError(
            f"{table_name}에 다음 컬럼이 없습니다: {missing_columns}"
        )

    # ID 컬럼: 실제 문자열 값으로 만든 후 object 타입으로 변환
    for column in object_columns:
        df[column] = (
            df[column]
            .astype("string")
            .astype("object")
        )

    # 날짜 컬럼: datetime으로 변환
    for column in datetime_columns:
        original_values = df[column].copy()

        converted_values = pd.to_datetime(
            original_values,
            errors="coerce"
        )

        invalid_mask = (
            original_values.notna()
            & converted_values.isna()
        )

        invalid_count = int(invalid_mask.sum())

        df[column] = converted_values

        print(
            f"[{table_name}] {column}: "
            f"datetime 변환 실패 {invalid_count:,}건"
        )

        if invalid_count > 0:
            print(
                "변환 실패 예시:",
                original_values[invalid_mask].head().tolist()
            )

    return df
# 1. 프로젝트 루트 찾기
for candidate in (Path.cwd(), *Path.cwd().parents):
    if (candidate / "src" / "utils" / "db.py").is_file():
        PROJECT_ROOT = candidate
        break
else:
    raise FileNotFoundError(
        "프로젝트 폴더 안에서 노트북을 실행하세요."
    )

# 2. 프로젝트 모듈 경로 등록
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# 3. 현재 .env 설정 반영
load_dotenv(PROJECT_ROOT / ".env", override=True)

# 4. 설정 반영 후 DB 연결 함수 불러오기
from src.utils.db import get_engine

print("프로젝트 루트:", PROJECT_ROOT)


프로젝트 루트: /Users/apple/Documents/team4-final-project


In [249]:
engine = get_engine()

try:
    with engine.connect() as connection:
        display(pd.read_sql_query(
            text("SELECT DATABASE() AS `현재 연결된 데이터베이스`"),
            connection
        ))

        display(pd.read_sql_query(
            text("SELECT * FROM final.polls_question LIMIT 5"),
            connection
        ))
finally:
    engine.dispose()

,현재 연결된 데이터베이스
0,final


,id,question_text,created_at
0,99,가장 신비한 매력이 있는 사람은?,2023-03-31 15:22:53
1,100,"""이 사람으로 한 번 살아보고 싶다"" 하는 사람은?",2023-03-31 15:22:53
2,101,미래의 틱톡커는?,2023-03-31 15:22:54
3,102,여기서 제일 특이한 친구는?,2023-03-31 15:22:54
4,103,가장 지켜주고 싶은 사람은?,2023-03-31 15:22:55


## 2. 데이터 불러오기

원본 테이블과 조인 확인에 필요한 키를 불러옵니다. 미래 시간의 기준은 DB의 `NOW()`입니다.


In [250]:
engine = get_engine()
try:
    with engine.connect() as connection:
        df = pd.read_sql_query(
            text("SELECT * FROM polls_question"),
            connection,
            dtype={"id": "Int64"},
        )
        # SQL의 NOW()와 같은 기준으로 미래 시간을 확인합니다.
        checked_at = pd.Timestamp(pd.read_sql_query(
            text("SELECT NOW() AS checked_at"), connection
        ).at[0, "checked_at"])
finally:
    engine.dispose()
df = convert_column_types(
    df=df,
    object_columns=["id"],
    datetime_columns=["created_at"],
    table_name="polls_question"
)

# 변환 결과 확인
display(df[["id", "created_at"]].dtypes)


print(
    f"polls_question: {len(df):,}행 "
    f"/ DB 기준 확인 시간: {checked_at}"
)

display(df.head())



[polls_question] created_at: datetime 변환 실패 0건


id                    object
created_at    datetime64[us]
dtype: object

polls_question: 5,025행 / DB 기준 확인 시간: 2026-09-08 23:57:57


,id,question_text,created_at
0,99,가장 신비한 매력이 있는 사람은?,2023-03-31 15:22:53
1,100,"""이 사람으로 한 번 살아보고 싶다"" 하는 사람은?",2023-03-31 15:22:53
2,101,미래의 틱톡커는?,2023-03-31 15:22:54
3,102,여기서 제일 특이한 친구는?,2023-03-31 15:22:54
4,103,가장 지켜주고 싶은 사람은?,2023-03-31 15:22:55


## 3. 결측치 확인

NULL과 앞뒤 일반 공백을 제거한 빈 문자열을 구분합니다.


In [251]:
trimmed_text = df["question_text"].astype("string").str.strip(" ")
text_length = trimmed_text.str.len()
mean_length = text_length.mean()
created_at = pd.to_datetime(df["created_at"], errors="raise")

missing = pd.DataFrame([{
    "전체 행 수": len(df),
    "질문 ID 결측 행 수": df["id"].isna().sum(),
    "질문 내용 NULL 행 수": df["question_text"].isna().sum(),
    "질문 내용 공백 행 수": trimmed_text.eq("").fillna(False).sum(),
    "생성 시간 결측 행 수": df["created_at"].isna().sum(),
}])

display(missing.head(20))
if len(missing) > 20:
    print(f'전체 {len(missing):,}행 중 앞 20행 표시')


,전체 행 수,질문 ID 결측 행 수,질문 내용 NULL 행 수,질문 내용 공백 행 수,생성 시간 결측 행 수
0,5025,0,0,0,0


## 4. 이상치 확인

질문 길이와 ID 범위, DB의 NOW() 기준 미래 생성 시간을 확인합니다.


In [252]:
outliers = pd.DataFrame([{
    "최소 질문 글자 수": text_length.min(),
    "평균 질문 글자 수": round(mean_length, 2) if pd.notna(mean_length) else None,
    "최대 질문 글자 수": text_length.max(),
    "최초 질문 생성 시간": created_at.min(),
    "최근 질문 생성 시간": created_at.max(),
    "미래 생성 시간 행 수": created_at.gt(checked_at).sum(),
}])

display(outliers.head(20))
if len(outliers) > 20:
    print(f'전체 {len(outliers):,}행 중 앞 20행 표시')

,최소 질문 글자 수,평균 질문 글자 수,최대 질문 글자 수,최초 질문 생성 시간,최근 질문 생성 시간,미래 생성 시간 행 수
0,3,19.41,45,2023-03-31 15:22:53,2023-06-06 06:15:52,0


## 5. 질문 ID 중복 확인

동일한 ID가 두 행 이상에 사용되었는지 확인합니다.


In [253]:
id_counts = df.groupby("id", dropna=False).size().reset_index(name="중복 행 수")
duplicate_ids = (
    id_counts.loc[id_counts["중복 행 수"].gt(1)]
    .rename(columns={"id": "중복 질문 ID"})
    .sort_values("중복 행 수", ascending=False)
    .reset_index(drop=True)
)

display(duplicate_ids.head(20))
if len(duplicate_ids) > 20:
    print(f'전체 {len(duplicate_ids):,}행 중 앞 20행 표시')


,중복 질문 ID,중복 행 수


## 6. 질문 내용 중복 확인

앞뒤 일반 공백 제거와 소문자 변환 후 비교합니다. MySQL의 collation에 따라 SQL 중복 결과와 차이가 날 수 있습니다.


In [254]:
valid_text = trimmed_text.notna() & trimmed_text.ne("").fillna(False)
text_rows = df.loc[valid_text, ["id"]].copy()
text_rows["정규화한 질문 내용"] = trimmed_text.loc[valid_text].str.lower()
# SQL의 GROUP_CONCAT(id ORDER BY id)에 해당합니다. ID를 먼저 정렬합니다.
text_counts = (
    text_rows.sort_values("id")
    .groupby("정규화한 질문 내용", dropna=False)
    .agg(**{
        "중복 질문 수": ("id", "size"),
        "해당 질문 ID 목록": (
            "id", lambda ids: ",".join(ids.dropna().astype("string"))
        ),
    })
    .reset_index()
)
duplicate_text = (
    text_counts.loc[text_counts["중복 질문 수"].gt(1)]
    .sort_values("중복 질문 수", ascending=False)
    .reset_index(drop=True)
)

display(duplicate_text.head(20))
if len(duplicate_text) > 20:
    print(f'전체 {len(duplicate_text):,}행 중 앞 20행 표시')


,정규화한 질문 내용,중복 질문 수,해당 질문 ID 목록
0,vote,56,"1139,1219,1232,1320,1507,1591,1651,1652,1653,1..."
1,지금 뭐하는지 궁금한 친구,3,"1951,3547,4530"
2,2세가 가장 귀여울 것 같은 사람은?,3,"1391,287,3160"
3,같이 밥먹고 싶은 사람은?,3,"1029,3226,4209"
4,인생 2회차인 것 같은 사람은?,3,"2098,3511,4494"
5,눈이 제일 큰 사람은?,3,"1397,2750,838"
6,용돈 제일 많이 받을 것 같은 사람,2,"1342,2188"
7,우리 집에 초대해서 야식 먹고 싶은 사람!,2,"3519,4502"
8,예능 나가면 한 건 할 것 같은 사람은?,2,"3512,4495"
9,오글거리는 말 제일 못할 것 같은 사람은?,2,"3570,4553"


전체 1,067행 중 앞 20행 표시


# polls_questionpiece 기초 전처리

삭제 전 품질과 조인 결과를 확인하고, `is_voted = 1 AND is_skipped = 1`인 행을 분석용 데이터에서 제외한 뒤 CSV로 저장합니다.



## 1. 공통 준비

현재 실행 폴더부터 상위 폴더를 확인해 프로젝트 루트를 찾고 팀의 DB 연결 함수를 불러옵니다.


In [256]:
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

def find_project_root():
    """실행 위치에 관계없이 src/utils/db.py가 있는 프로젝트 루트를 찾습니다."""
    starts = []
    if "__file__" in globals():
        starts.append(Path(__file__).resolve().parent)
    starts.append(Path.cwd())
    for start in starts:
        for candidate in (start, *start.parents):
            if (candidate / "src" / "utils" / "db.py").is_file():
                return candidate
    raise FileNotFoundError(
        "src/utils/db.py를 찾지 못했습니다. 이 파일을 팀 저장소 안에 넣고 실행하세요."
    )

PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from sqlalchemy import text
from src.utils.db import get_engine

print(f'프로젝트 루트: {PROJECT_ROOT}')


프로젝트 루트: /Users/apple/Documents/team4-final-project


## 2. 데이터 불러오기

원본 테이블과 조인 확인에 필요한 키를 불러옵니다. 미래 시간의 기준은 DB의 `NOW()`입니다.


In [257]:
engine = get_engine()
try:
    with engine.connect() as connection:
        df = pd.read_sql_query(
            text("SELECT * FROM polls_questionpiece"), connection,
            dtype={"id": "Int64", "question_id": "Int64",
                   "is_voted": "Int64", "is_skipped": "Int64"},
        )
        question_ids = pd.read_sql_query(
            text("SELECT id FROM polls_question"), connection,
            dtype={"id": "Int64"},
        )["id"]
        # 첨부 SQL의 WHERE question_id IS NOT NULL을 실제 연결 키 기준으로 수정.
        # 기록 테이블 전체 컬럼 대신 확인에 필요한 키만 읽습니다.
        recorded_piece_ids = pd.read_sql_query(text("""
            SELECT DISTINCT question_piece_id
            FROM accounts_userquestionrecord
            WHERE question_piece_id IS NOT NULL
        """), connection, dtype={"question_piece_id": "Int64"})["question_piece_id"]
        checked_at = pd.Timestamp(pd.read_sql_query(
            text("SELECT NOW() AS checked_at"), connection
        ).at[0, "checked_at"])
finally:
    engine.dispose()
# 데이터 타입 변환
questionpiece_df = convert_column_types(
    df=df,
    object_columns=["id", "question_id"],
    datetime_columns=["created_at"],
    table_name="polls_questionpiece"
)

display(questionpiece_df.dtypes)

print(f'polls_questionpiece: {len(df):,}행 / DB 기준 확인 시간: {checked_at}')
display(df.head())


[polls_questionpiece] created_at: datetime 변환 실패 0건


id                     object
is_voted                Int64
created_at     datetime64[us]
question_id            object
is_skipped              Int64
dtype: object

polls_questionpiece: 1,265,476행 / DB 기준 확인 시간: 2026-09-08 23:58:09


,id,is_voted,created_at,question_id,is_skipped
0,998458,1,2023-04-28 12:27:22,252,0
1,998459,1,2023-04-28 12:27:22,244,0
2,998460,1,2023-04-28 12:27:22,183,0
3,998461,1,2023-04-28 12:27:22,101,0
4,998462,1,2023-04-28 12:27:22,209,0


## 3. 제외 조건 정의

선택한 조건에 해당하는 행만 제외하며, 결측 행을 별도로 삭제하는 조건은 추가하지 않습니다.


In [258]:
def exclusion_mask(df):
    """두 상태가 모두 1인 행만 제외합니다. 상태 결측 행은 별도로 삭제하지 않습니다."""
    return (df["is_voted"].eq(1) & df["is_skipped"].eq(1)).fillna(False)

def clean_data(df):
    """선택한 제외 조건만 적용하고, 나머지 컬럼과 값은 그대로 유지합니다."""
    return df.loc[~exclusion_mask(df)].copy()


## 4. 결측 확인

질문 단위 ID, 질문 ID, 투표·스킵 여부, 생성 시간의 결측을 확인합니다.


In [259]:
created_at = pd.to_datetime(df["created_at"], errors="raise")
missing = pd.DataFrame([{
    "전체 행 수": len(df),
    "질문 단위 ID 결측 행 수": df["id"].isna().sum(),
    "질문 ID 결측 행 수": df["question_id"].isna().sum(),
    "투표 여부 결측 행 수": df["is_voted"].isna().sum(),
    "스킵 여부 결측 행 수": df["is_skipped"].isna().sum(),
    "생성 시간 결측 행 수": df["created_at"].isna().sum(),
}])

display(missing.head(20))
if len(missing) > 20:
    print(f'전체 {len(missing):,}행 중 앞 20행 표시')


,전체 행 수,질문 단위 ID 결측 행 수,질문 ID 결측 행 수,투표 여부 결측 행 수,스킵 여부 결측 행 수,생성 시간 결측 행 수
0,1265476,0,0,0,0,0


## 5. 투표·스킵 상태 조합

두 상태의 조합별 질문 단위 수를 확인합니다.


In [260]:
flags = ["is_voted", "is_skipped"]
flag_names = {"is_voted": "투표 여부", "is_skipped": "스킵 여부"}
combinations = (
    df.groupby(flags, dropna=False, sort=True)
    .size().reset_index(name="해당 조합 행 수")
    .rename(columns=flag_names)
)

display(combinations.head(20))
if len(combinations) > 20:
    print(f'전체 {len(combinations):,}행 중 앞 20행 표시')


,투표 여부,스킵 여부,해당 조합 행 수
0,0,0,46789
1,1,0,1217560
2,1,1,1127


## 6. 이상치 확인

ID·상태 값과 미래 시간을 확인하고, 투표·스킵 동시 기록을 집계합니다.


In [261]:
outliers = pd.DataFrame([{
    "전체 행 수": len(df),
    # SQL의 NOT IN과 같이 NULL은 비정상값 수에 중복 집계하지 않습니다.
    "투표 여부 비정상값 수": (
        df["is_voted"].notna() & ~df["is_voted"].isin([0, 1])
    ).sum(),
    "스킵 여부 비정상값 수": (
        df["is_skipped"].notna() & ~df["is_skipped"].isin([0, 1])
    ).sum(),
    "투표와 스킵이 모두 기록된 행 수": exclusion_mask(df).sum(),
    "미래 생성 시간 행 수": created_at.gt(checked_at).sum(),
    "최초 생성 시간": created_at.min(),
    "최근 생성 시간": created_at.max(),
}])

display(outliers.head(20))
if len(outliers) > 20:
    print(f'전체 {len(outliers):,}행 중 앞 20행 표시')


,전체 행 수,투표 여부 비정상값 수,스킵 여부 비정상값 수,투표와 스킵이 모두 기록된 행 수,미래 생성 시간 행 수,최초 생성 시간,최근 생성 시간
0,1265476,0,0,1127,0,2023-04-28 12:27:22,2024-05-07 11:32:30


## 7. 질문 단위 ID 중복 확인

동일 ID의 반복 여부를 확인합니다.


In [262]:
id_counts = df.groupby("id", dropna=False).size().reset_index(name="중복 행 수")
duplicate_ids = (
    id_counts.loc[id_counts["중복 행 수"].gt(1)]
    .rename(columns={"id": "중복 질문 단위 ID"})
    .sort_values("중복 행 수", ascending=False)
    .reset_index(drop=True)
)

display(duplicate_ids.head(20))
if len(duplicate_ids) > 20:
    print(f'전체 {len(duplicate_ids):,}행 중 앞 20행 표시')


,중복 질문 단위 ID,중복 행 수


## 8. 질문 내용 연결 확인

polls_questionpiece.question_id와 polls_question.id를 연결합니다.


In [263]:
question_keys = question_ids.dropna().drop_duplicates()
unlinked = df["question_id"].notna() & ~df["question_id"].isin(question_keys)
question_join = pd.DataFrame([{
    "질문 내용과 연결되지 않는 행 수": unlinked.sum(),
}])

display(question_join.head(20))
if len(question_join) > 20:
    print(f'전체 {len(question_join):,}행 중 앞 20행 표시')


,질문 내용과 연결되지 않는 행 수
0,1265476


## 9. 연결되지 않는 질문 ID 목록

질문 내용 테이블에 없는 질문 ID별 행 수를 확인합니다.


In [264]:
unlinked_questions = (
    df.loc[unlinked].groupby("question_id", dropna=False)
    .size().reset_index(name="해당 질문 ID의 행 수")
    .rename(columns={"question_id": "연결되지 않는 질문 ID"})
    .sort_values("해당 질문 ID의 행 수", ascending=False)
    .reset_index(drop=True)
)

display(unlinked_questions.head(20))
if len(unlinked_questions) > 20:
    print(f'전체 {len(unlinked_questions):,}행 중 앞 20행 표시')


,연결되지 않는 질문 ID,해당 질문 ID의 행 수
0,170,2030
1,290,2021
2,169,2017
3,211,2009
4,186,1991
5,310,1985
6,244,1975
7,102,1975
8,309,1963
9,152,1956


전체 4,944행 중 앞 20행 표시


## 10. 상태별 투표 기록 연결 확인

polls_questionpiece.id와 accounts_userquestionrecord.question_piece_id를 연결합니다. 기록이 여러 개여도 질문 단위를 한 번만 집계합니다.


In [265]:
record_keys = recorded_piece_ids.dropna().drop_duplicates()
joined = df[flags].copy()
joined["record_linked"] = df["id"].notna() & df["id"].isin(record_keys)
record_join = (
    joined.groupby(flags, dropna=False, sort=True)
    .agg(**{
        "전체 질문 단위 수": ("record_linked", "size"),
        "투표 기록이 연결된 질문 단위 수": ("record_linked", "sum"),
    })
    .reset_index().rename(columns=flag_names)
)
record_join["투표 기록이 연결되지 않은 질문 단위 수"] = (
    record_join["전체 질문 단위 수"] - record_join["투표 기록이 연결된 질문 단위 수"]
)

display(record_join.head(20))
if len(record_join) > 20:
    print(f'전체 {len(record_join):,}행 중 앞 20행 표시')


,투표 여부,스킵 여부,전체 질문 단위 수,투표 기록이 연결된 질문 단위 수,투표 기록이 연결되지 않은 질문 단위 수
0,0,0,46789,0,46789
1,1,0,1217560,0,1217560
2,1,1,1127,0,1127


## 확인 결과 모아두기

각 셀의 전체 확인 결과는 `results`에 보관됩니다. 앞 20행 표시 제한은 계산 결과에 영향을 주지 않습니다.


In [266]:
results = {'결측 확인': missing, '투표·스킵 상태 조합': combinations, '이상치 확인': outliers, '질문 단위 ID 중복 확인': duplicate_ids, '질문 내용 연결 확인': question_join, '연결되지 않는 질문 ID 목록': unlinked_questions, '상태별 투표 기록 연결 확인': record_join}


## 행 제외 및 전처리 결과 확인

원본 `df`에서 제외 조건을 적용한 `clean_df`를 만들고, 제외 전·후 행 수와 남은 대상 행 수를 확인합니다.


In [267]:
clean_df = clean_data(df)
removed_count = len(df) - len(clean_df)
summary = pd.DataFrame([{
    "삭제 전 행 수": len(df),
    "투표·스킵 동시 기록 제외 행 수": removed_count,
    "제외 비율(%)": round(removed_count * 100 / len(df), 2) if len(df) else None,
    "삭제 후 행 수": len(clean_df),
    "남아 있는 동시 기록 행 수": exclusion_mask(clean_df).sum(),
}])

results['전처리 결과'] = summary
display(summary)


,삭제 전 행 수,투표·스킵 동시 기록 제외 행 수,제외 비율(%),삭제 후 행 수,남아 있는 동시 기록 행 수
0,1265476,1127,0.09,1264349,0


# polls_questionset 기초 전처리

삭제 전 품질과 조인 결과를 확인하고, 같은 행에서 `opening_time < created_at`인 경우를 분석용 데이터에서 제외한 뒤 CSV로 저장합니다.



## 1. 공통 준비

현재 실행 폴더부터 상위 폴더를 확인해 프로젝트 루트를 찾고 팀의 DB 연결 함수를 불러옵니다.


In [269]:
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

def find_project_root():
    """실행 위치에 관계없이 src/utils/db.py가 있는 프로젝트 루트를 찾습니다."""
    starts = []
    if "__file__" in globals():
        starts.append(Path(__file__).resolve().parent)
    starts.append(Path.cwd())
    for start in starts:
        for candidate in (start, *start.parents):
            if (candidate / "src" / "utils" / "db.py").is_file():
                return candidate
    raise FileNotFoundError(
        "src/utils/db.py를 찾지 못했습니다. 이 파일을 팀 저장소 안에 넣고 실행하세요."
    )

PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from sqlalchemy import text
from src.utils.db import get_engine

print(f'프로젝트 루트: {PROJECT_ROOT}')


프로젝트 루트: /Users/apple/Documents/team4-final-project


## 2. 데이터 불러오기

원본 테이블과 조인 확인에 필요한 키를 불러옵니다. 미래 시간의 기준은 DB의 `NOW()`입니다.


In [270]:
engine = get_engine()
try:
    with engine.connect() as connection:
        df = pd.read_sql_query(
            text("SELECT * FROM polls_questionset"), connection,
            dtype={"id": "Int64", "user_id": "Int64"},
        )
        user_ids = pd.read_sql_query(
            text("SELECT id FROM accounts_user"), connection,
            dtype={"id": "Int64"},
        )["id"]
        checked_at = pd.Timestamp(pd.read_sql_query(
            text("SELECT NOW() AS checked_at"), connection
        ).at[0, "checked_at"])
finally:
    engine.dispose()
questionset_df = convert_column_types(
    df=df,
    object_columns=["id","user_id"],
    datetime_columns=["opening_time", "created_at"],
    table_name="polls_questionset"
)

display(questionset_df.dtypes)

print(f'polls_questionset: {len(df):,}행 / DB 기준 확인 시간: {checked_at}')
display(df.head())


[polls_questionset] opening_time: datetime 변환 실패 0건
[polls_questionset] created_at: datetime 변환 실패 0건


id                                object
question_piece_id_list               str
opening_time              datetime64[us]
status                               str
created_at                datetime64[us]
user_id                           object
dtype: object

polls_questionset: 158,384행 / DB 기준 확인 시간: 2026-09-08 23:58:16


,id,question_piece_id_list,opening_time,status,created_at,user_id
0,99817,"[998458, 998459, 998460, 998461, 998462, 99846...",2023-04-28 12:27:22,F,2023-04-28 12:27:23,849436
1,99830,"[998588, 998589, 998590, 998591, 998592, 99859...",2023-04-28 12:28:07,F,2023-04-28 12:28:07,849438
2,99840,"[998689, 998691, 998693, 998695, 998697, 99869...",2023-04-28 12:28:38,F,2023-04-28 12:28:38,847375
3,99841,"[998688, 998690, 998692, 998694, 998696, 99869...",2023-04-28 12:28:38,F,2023-04-28 12:28:38,849446
4,99848,"[998768, 998769, 998770, 998771, 998772, 99877...",2023-04-28 12:28:57,F,2023-04-28 12:28:57,849477


## 3. 제외 조건 정의

선택한 조건에 해당하는 행만 제외하며, 결측 행을 별도로 삭제하는 조건은 추가하지 않습니다.


In [271]:
def exclusion_mask(df):
    """오픈 시간이 더 이른 행만 제외합니다. 동일 시간 또는 결측 시간은 제외하지 않습니다."""
    created_at = pd.to_datetime(df["created_at"], errors="raise")
    opening_time = pd.to_datetime(df["opening_time"], errors="raise")
    return (
        created_at.notna() & opening_time.notna() & opening_time.lt(created_at)
    ).fillna(False)

def clean_data(df):
    """선택한 제외 조건만 적용하고, 나머지 컬럼과 값은 그대로 유지합니다."""
    return df.loc[~exclusion_mask(df)].copy()


## 4. 결측 확인

NULL, 공백 문자열, 빈 배열 형태의 질문 목록을 구분합니다.


In [272]:
piece_list = df["question_piece_id_list"].astype("string")
created_at = pd.to_datetime(df["created_at"], errors="raise")
opening_time = pd.to_datetime(df["opening_time"], errors="raise")
missing = pd.DataFrame([{
    "전체 행 수": len(df),
    "질문 세트 ID 결측 행 수": df["id"].isna().sum(),
    "질문 목록 NULL 행 수": df["question_piece_id_list"].isna().sum(),
    "질문 목록 공백 행 수": piece_list.str.strip(" ").eq("").fillna(False).sum(),
    "질문 목록이 빈 배열인 행 수": (
        piece_list.str.replace(" ", "", regex=False).eq("[]").fillna(False).sum()
    ),
    "질문 오픈 시간 결측 행 수": df["opening_time"].isna().sum(),
    "상태 결측 행 수": df["status"].isna().sum(),
    "생성 시간 결측 행 수": df["created_at"].isna().sum(),
    "사용자 ID 결측 행 수": df["user_id"].isna().sum(),
}])

display(missing.head(20))
if len(missing) > 20:
    print(f'전체 {len(missing):,}행 중 앞 20행 표시')


,전체 행 수,질문 세트 ID 결측 행 수,질문 목록 NULL 행 수,질문 목록 공백 행 수,질문 목록이 빈 배열인 행 수,질문 오픈 시간 결측 행 수,상태 결측 행 수,생성 시간 결측 행 수,사용자 ID 결측 행 수
0,158384,0,0,0,0,0,0,0,0


## 5. 이상치 확인

ID·미래 생성 시간과 생성·오픈 시간의 범위를 확인합니다.


In [273]:
outliers = pd.DataFrame([{
    "전체 행 수": len(df),
    "미래 생성 시간 행 수": created_at.gt(checked_at).sum(),
    "최초 생성 시간": created_at.min(),
    "최근 생성 시간": created_at.max(),
    "최소 질문 오픈 시간": opening_time.min(),
    "최대 질문 오픈 시간": opening_time.max(),
}])

display(outliers.head(20))
if len(outliers) > 20:
    print(f'전체 {len(outliers):,}행 중 앞 20행 표시')


,전체 행 수,미래 생성 시간 행 수,최초 생성 시간,최근 생성 시간,최소 질문 오픈 시간,최대 질문 오픈 시간
0,158384,0,2023-04-28 12:27:23,2024-05-07 11:32:30,2023-04-28 12:27:22,2024-05-07 12:12:30


## 6. 질문 세트 상태 분포

상태 코드별 행 수를 확인합니다.


In [274]:
statuses = (
    df.groupby("status", dropna=False).size().reset_index(name="해당 상태 행 수")
    .rename(columns={"status": "질문 세트 상태"})
    .sort_values("해당 상태 행 수", ascending=False)
    .reset_index(drop=True)
)

display(statuses.head(20))
if len(statuses) > 20:
    print(f'전체 {len(statuses):,}행 중 앞 20행 표시')


,질문 세트 상태,해당 상태 행 수
0,F,153411
1,O,4407
2,C,566


## 7. 질문 세트 ID 중복 확인

동일 ID의 반복 여부를 확인합니다.


In [275]:
id_counts = df.groupby("id", dropna=False).size().reset_index(name="동일 ID 행 수")
duplicate_ids = (
    id_counts.loc[id_counts["동일 ID 행 수"].gt(1)]
    .rename(columns={"id": "중복 질문 세트 ID"})
    .sort_values("동일 ID 행 수", ascending=False)
    .reset_index(drop=True)
)

display(duplicate_ids.head(20))
if len(duplicate_ids) > 20:
    print(f'전체 {len(duplicate_ids):,}행 중 앞 20행 표시')


,중복 질문 세트 ID,동일 ID 행 수


## 8. 동일 내용 중복 확인

ID를 제외한 다섯 컬럼이 같은 행을 확인합니다. 문자열 비교는 pandas 기준입니다.


In [276]:
content_columns = [
    "user_id", "question_piece_id_list", "opening_time", "status", "created_at"
]
content = df[content_columns].copy()
content["question_piece_id_list"] = piece_list
# SQL과 같이 NULL을 포함한 그룹도 확인합니다. 목록의 순서는 바꾸지 않습니다.
# 문자열 동일성은 pandas 기준이며 MySQL collation에 따라 차이가 날 수 있습니다.
content_counts = (
    content.groupby(content_columns, dropna=False)
    .size().reset_index(name="동일 내용 행 수")
)
duplicate_content = (
    content_counts.loc[content_counts["동일 내용 행 수"].gt(1)]
    .rename(columns={
        "user_id": "사용자 ID", "question_piece_id_list": "질문 단위 ID 목록",
        "opening_time": "질문 오픈 시간", "status": "질문 세트 상태",
        "created_at": "생성 시간",
    })
    .sort_values("동일 내용 행 수", ascending=False)
    .reset_index(drop=True)
)

display(duplicate_content.head(20))
if len(duplicate_content) > 20:
    print(f'전체 {len(duplicate_content):,}행 중 앞 20행 표시')


,사용자 ID,질문 단위 ID 목록,질문 오픈 시간,질문 세트 상태,생성 시간,동일 내용 행 수


## 9. 사용자 테이블 연결 확인

polls_questionset.user_id와 accounts_user.id의 연결 여부를 확인합니다.


In [277]:
user_keys = user_ids.dropna().drop_duplicates()
unlinked = df["user_id"].notna() & ~df["user_id"].isin(user_keys)
user_join = pd.DataFrame([{
    "사용자 테이블과 연결되지 않는 세트 행 수": unlinked.sum(),
    "연결되지 않는 사용자 ID 수": df.loc[unlinked, "user_id"].nunique(),
}])

display(user_join.head(20))
if len(user_join) > 20:
    print(f'전체 {len(user_join):,}행 중 앞 20행 표시')


,사용자 테이블과 연결되지 않는 세트 행 수,연결되지 않는 사용자 ID 수
0,158384,4972


## 10. 시간 선후 관계 확인

같은 행에서 오픈 시간이 생성 시간보다 빠른 경우의 행 수와 전체 대비 비율을 확인합니다.


In [278]:
early_count = exclusion_mask(df).sum()
time_order = pd.DataFrame([{
    "전체 행 수": len(df),
    "오픈 시간이 생성 시간보다 빠른 행 수": early_count,
    "전체 대비 비율(%)": round(early_count * 100 / len(df), 2) if len(df) else None,
}])

display(time_order.head(20))
if len(time_order) > 20:
    print(f'전체 {len(time_order):,}행 중 앞 20행 표시')


,전체 행 수,오픈 시간이 생성 시간보다 빠른 행 수,전체 대비 비율(%)
0,158384,679,0.43


## 확인 결과 모아두기

각 셀의 전체 확인 결과는 `results`에 보관됩니다. 앞 20행 표시 제한은 계산 결과에 영향을 주지 않습니다.


In [279]:
results = {'결측 확인': missing, '이상치 확인': outliers, '질문 세트 상태 분포': statuses, '질문 세트 ID 중복 확인': duplicate_ids, '동일 내용 중복 확인': duplicate_content, '사용자 테이블 연결 확인': user_join, '시간 선후 관계 확인': time_order}


## 행 제외 및 전처리 결과 확인

원본 `df`에서 제외 조건을 적용한 `clean_df`를 만들고, 제외 전·후 행 수와 남은 대상 행 수를 확인합니다.


In [280]:
clean_df = clean_data(df)
removed_count = len(df) - len(clean_df)
summary = pd.DataFrame([{
    "삭제 전 행 수": len(df),
    "오픈 시간이 더 이른 제외 행 수": removed_count,
    "제외 비율(%)": round(removed_count * 100 / len(df), 2) if len(df) else None,
    "삭제 후 행 수": len(clean_df),
    "남아 있는 시간 역전 행 수": exclusion_mask(clean_df).sum(),
}])

results['전처리 결과'] = summary
display(summary)


,삭제 전 행 수,오픈 시간이 더 이른 제외 행 수,제외 비율(%),삭제 후 행 수,남아 있는 시간 역전 행 수
0,158384,679,0.43,157705,0
